# The Generalized Reduced Gradient (GRG) Algorithm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Bayoumy/MECH559/blob/DEV/L09/L09_generalized_reduced_gradient.ipynb)
### MECH 559 - Systems Optimization - L09

Companion notebook to the "Algorithms for nonlinearly constrained optimization" lecture. GRG turns an equality-constrained problem into an *unconstrained* one in the decision variables by eliminating the state variables through the constraint, then takes a descent step in decision space, predicts the state move with a first-order (linear) correction, and finally restores exact feasibility with Newton-Raphson - exactly the three-move picture ("Geometrical perspective") in the slides.

## Learning objectives

By the end, students should be able to:

1. Split $x$ into decision variables $x_d$ and state variables $x_s$ and compute the reduced gradient $\nabla w$.
2. Implement the three GRG moves: the decision-space step, the linearized state-space prediction, and the Newton-Raphson feasibility correction.
3. Recognize why the *predicted* state update $\hat x_{s_{k+1}}$ generally violates $h=0$, and why an inner Newton loop is required to fix it.
4. Run the full GRG iteration to convergence and cross-check the result against a direct constrained solve.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

COLORS = {"decision": "#1D5FA8", "state": "#2A9D5C", "correction": "#D1495B",
          "feasible": "#A8DADC", "optimum": "#D1495B", "accent": "#EDAE49"}

print(f"NumPy {np.__version__}, SciPy ok")

NumPy 2.5.0, SciPy ok


---
## 1. The problem

$$
\min_{x_1,x_2}\; f(x_1,x_2) = (x_1-2)^2+(x_2-2)^2
\qquad\text{s.t.}\qquad
h(x_1,x_2) = x_1^2+x_2^2-1 = 0.
$$

The constraint is the unit circle; $f$ is the squared distance to $(2,2)$, so geometrically we want the point on the circle closest to $(2,2)$. By symmetry that is $x^\star=(1/\sqrt2,\,1/\sqrt2)\approx(0.7071,0.7071)$ with $f(x^\star)=(2-\sqrt2/... )$ - we verify this numerically below rather than trust the by-eye argument.

We choose $x_d=x_1$ as the **decision** variable and $x_s=x_2$ as the **state** variable (valid as long as $\nabla_{x_s}h=2x_2\neq0$, i.e. away from $x_2=0$, which holds on the entire upper semicircle we work on).

In [2]:
def f(x):
    return (x[0] - 2.0) ** 2 + (x[1] - 2.0) ** 2

def grad_f(x):
    return np.array([2.0 * (x[0] - 2.0), 2.0 * (x[1] - 2.0)])

def h(x):
    return x[0] ** 2 + x[1] ** 2 - 1.0

def dh_dxd(x):   # d h / d x1
    return 2.0 * x[0]

def dh_dxs(x):   # d h / d x2
    return 2.0 * x[1]

def reduced_gradient(x):
    gfull = grad_f(x)
    gfd, gfs = gfull[0], gfull[1]
    return gfd - gfs * (1.0 / dh_dxs(x)) * dh_dxd(x)

x_probe = np.array([0.0, 1.0])
print(f"x_probe = {x_probe}, h(x_probe) = {h(x_probe):.3e}, reduced gradient nabla w = {reduced_gradient(x_probe):.4f}")

x_probe = [0. 1.], h(x_probe) = 0.000e+00, reduced gradient nabla w = -4.0000


---
## 2. Anatomy of one GRG step

Starting from a feasible point $x_k$ on the circle, one GRG iteration is:

1. **Decision move** (blue): $x_{d_{k+1}} = x_{d_k} - \alpha_k (\nabla w)_k$.
2. **State prediction** (green), a *first-order* estimate from linearizing $h$:
   $\hat x_{s_{k+1}} = x_{s_k} - (\nabla_{x_s}h)^{-1}_k(\nabla_{x_d}h)_k(\partial x_d)_k$.
3. **Newton-Raphson correction** (red): $\hat x_{s_{k+1}}$ generally does not satisfy $h=0$ exactly (the update used a linear approximation), so we run
   $[x_{s_{k+1}}]_{i+1} = [x_{s_{k+1}}]_i - (\nabla_{x_s}h)^{-1}\,h(x_{d_{k+1}}, [x_{s_{k+1}}]_i)$, $[x_{s_{k+1}}]_0=\hat x_{s_{k+1}}$,
   until $|h|\le\epsilon$.

Let's visualize exactly these three moves for one step from $x_0=(0,1)$.

In [3]:
def state_newton_correction(x1_new, x2_guess, tol=1e-12, max_iter=50):
    x2 = x2_guess
    for _ in range(max_iter):
        residual = h([x1_new, x2])
        if abs(residual) < tol:
            break
        x2 -= residual / dh_dxs([x1_new, x2])
    return x2

x0 = np.array([0.0, 1.0])
alpha = 0.1
dw = reduced_gradient(x0)
d_xd = -alpha * dw                                  # (partial x_d)_0

# Move 1: decision space
x1_decision = x0[0] + d_xd

# Move 2: state space (linear prediction)
x2_hat = x0[1] - (1.0 / dh_dxs(x0)) * dh_dxd(x0) * d_xd
point_state = np.array([x1_decision, x2_hat])

# Move 3: Newton-Raphson correction back onto h=0
x2_corrected = state_newton_correction(x1_decision, x2_hat)
point_corrected = np.array([x1_decision, x2_corrected])

print(f"nabla w at x0        = {dw:.4f}")
print(f"decision step        : x1  {x0[0]:.4f} -> {x1_decision:.4f}")
print(f"state prediction     : x2  {x0[1]:.4f} -> {x2_hat:.4f}  (h = {h(point_state): .4e}, off the circle)")
print(f"Newton-Raphson fix   : x2  {x2_hat:.4f} -> {x2_corrected:.4f}  (h = {h(point_corrected): .4e})")

nabla w at x0        = -4.0000
decision step        : x1  0.0000 -> 0.4000
state prediction     : x2  1.0000 -> 1.0000  (h =  1.6000e-01, off the circle)
Newton-Raphson fix   : x2  1.0000 -> 0.9165  (h =  0.0000e+00)


In [4]:
theta = np.linspace(0, 2 * np.pi, 400)
circle = np.stack([np.cos(theta), np.sin(theta)])

x1g, x2g = np.meshgrid(np.linspace(-1.6, 2.6, 300), np.linspace(-1.6, 2.6, 300))
Fg = (x1g - 2.0) ** 2 + (x2g - 2.0) ** 2

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.contour(x1g, x2g, Fg, levels=25, cmap="Greys", alpha=0.5)
ax.plot(circle[0], circle[1], color="black", lw=1.5, label=r"$h(x)=0$")

ax.plot(*x0, "o", color="black", ms=9, label=r"$x_k$ (feasible)")
ax.annotate("", xy=(x1_decision, x0[1]), xytext=x0,
            arrowprops=dict(arrowstyle="->", color=COLORS["decision"], lw=2))
ax.plot(x1_decision, x0[1], "s", color=COLORS["decision"], ms=8, label="1. decision-space move")

ax.annotate("", xy=point_state, xytext=(x1_decision, x0[1]),
            arrowprops=dict(arrowstyle="->", color=COLORS["state"], lw=2))
ax.plot(*point_state, "^", color=COLORS["state"], ms=9, label=r"2. state-space move $\hat x_{s_{k+1}}$ (off circle)")

ax.annotate("", xy=point_corrected, xytext=point_state,
            arrowprops=dict(arrowstyle="->", color=COLORS["correction"], lw=2))
ax.plot(*point_corrected, "*", color=COLORS["correction"], ms=16, label=r"3. Newton-Raphson correction $x_{s_{k+1}}$")

ax.set(xlabel="$x_1$", ylabel="$x_2$", aspect="equal", title="One GRG iteration: decision, state, and correction moves")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

C:\Users\ahbay\AppData\Local\Temp\ipykernel_47836\2963645752.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. The full GRG loop

We now repeat the three moves, with a simple backtracking line search on $\alpha_k$ (accepting a step only if it decreases $f$ and the inner Newton-Raphson correction converges), until $|\nabla w|\le\epsilon$.

In [5]:
def grg(x0, tol=1e-6, max_iter=60, alpha0=1.0):
    x = np.array(x0, dtype=float)
    traj = [x.copy()]
    grad_hist = [abs(reduced_gradient(x))]
    converged = False
    for k in range(max_iter):
        dw = reduced_gradient(x)
        if abs(dw) < tol:
            converged = True
            break
        alpha = alpha0
        for _ in range(40):
            d_xd = -alpha * dw
            x1_new = x[0] + d_xd
            x2_hat = x[1] - (1.0 / dh_dxs(x)) * dh_dxd(x) * d_xd
            x2_new = state_newton_correction(x1_new, x2_hat)
            x_new = np.array([x1_new, x2_new])
            if np.isfinite(x_new).all() and f(x_new) < f(x) and abs(h(x_new)) < 1e-8:
                break
            alpha *= 0.5
        x = x_new
        traj.append(x.copy())
        grad_hist.append(abs(reduced_gradient(x)))
    return np.array(traj), np.array(grad_hist), converged

traj, grad_hist, converged = grg(x0=[0.0, 1.0])
x_star_analytic = np.array([1.0, 1.0]) / np.sqrt(2.0)

print(f"GRG {'converged' if converged else 'stopped (max_iter reached)'} after {len(traj) - 1} iterations, |nabla w| = {grad_hist[-1]:.2e}")
print(f"x_GRG      = {traj[-1]}")
print(f"x_analytic = {x_star_analytic}")
print(f"||x_GRG - x_analytic|| = {np.linalg.norm(traj[-1] - x_star_analytic):.2e}")
print(f"f(x_GRG) = {f(traj[-1]):.6f},  h(x_GRG) = {h(traj[-1]):.2e}")

GRG converged after 17 iterations, |nabla w| = 9.42e-07
x_GRG      = [0.70710686 0.7071067 ]
x_analytic = [0.70710678 0.70710678]
||x_GRG - x_analytic|| = 1.18e-07
f(x_GRG) = 3.343146,  h(x_GRG) = 6.37e-13


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].contour(x1g, x2g, Fg, levels=25, cmap="Greys", alpha=0.5)
axes[0].plot(circle[0], circle[1], color="black", lw=1.5, label=r"$h(x)=0$")
axes[0].plot(traj[:, 0], traj[:, 1], "-o", color=COLORS["decision"], ms=5, label="GRG iterates")
axes[0].plot(*x_star_analytic, "*", color=COLORS["optimum"], ms=18, label="analytic optimum")
axes[0].set(xlabel="$x_1$", ylabel="$x_2$", aspect="equal", title="GRG trajectory on the constraint curve")
axes[0].legend(fontsize=9)

axes[1].semilogy(grad_hist, "-o", color=COLORS["correction"])
axes[1].set(xlabel="iteration $k$", ylabel=r"$|\nabla w_k|$", title="Reduced-gradient norm (convergence)")

plt.tight_layout()
plt.show()

C:\Users\ahbay\AppData\Local\Temp\ipykernel_47836\3106158783.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Cross-check against a direct constrained solve

`scipy.optimize.minimize` with `SLSQP` solves the same problem directly (treating $x_1,x_2$ jointly rather than eliminating a state variable). Both approaches should agree to solver tolerance.

In [7]:
res = minimize(f, x0=[0.0, 1.0], constraints=[{"fun": h, "type": "eq"}], method="SLSQP")
print("SLSQP :", res.x, " f =", res.fun)
print("GRG   :", traj[-1], " f =", f(traj[-1]))
print("max coordinate discrepancy:", np.max(np.abs(res.x - traj[-1])))
assert np.allclose(res.x, traj[-1], atol=1e-5)
print("\nAgreement confirmed.")

SLSQP : [0.70710675 0.70710681]  f = 3.343145750463138
GRG   : [0.70710686 0.7071067 ]  f = 3.343145750506495
max coordinate discrepancy: 1.1208630179559975e-07

Agreement confirmed.


---
## Takeaways

1. GRG replaces the equality-constrained problem with an *unconstrained* one in $x_d$ by using $h=0$ to eliminate $x_s$ - but only *locally* and to *first order*, which is why the state move is only a prediction.
2. The predicted state $\hat x_{s_{k+1}}$ leaves the constraint surface because the linearization of $h$ is not exact; a Newton-Raphson inner loop is required at every outer iteration to restore feasibility.
3. The reduced gradient $\nabla w$ plays exactly the role of $\nabla f$ in an unconstrained descent method: the outer loop here is just steepest descent on $w(x_d)$, so it inherits steepest descent's linear convergence rate (see `L09_convergence_and_termination.ipynb`) - GRG can equally be driven by Newton or quasi-Newton updates in decision space for faster local convergence.
4. The three-color picture (decision / state / correction) reproduced above is exactly the "Geometrical perspective" figure in the slides.